In [0]:
from pyspark.sql import functions as F

t = spark.table("retail_inv.silver.sales_daily")

# Check for missing required fields and inconsistent availability flags.
quality = t.agg(
    F.sum(
        F.when(
            F.col("item_id").isNull()
            | F.col("store_id").isNull()
            | F.col("date").isNull()
            | F.col("units_demand").isNull(),
            1
        ).otherwise(0)
    ).alias("missing_required_rows"),

    F.sum(
        F.when(
            F.col("is_available") != F.col("sell_price").isNotNull(),
            1
        ).otherwise(0)
    ).alias("availability_flag_mismatches")
).first()

print("Rows with missing required fields:",
      f"{quality['missing_required_rows']:,}")
print("Availability flag mismatches:",
      f"{quality['availability_flag_mismatches']:,}")

# Check the intended item-store-date grain.
duplicate_keys = (
    t.groupBy("item_id", "store_id", "date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate item-store-date keys:", f"{duplicate_keys:,}")

Rows with missing required fields: 0
Availability flag mismatches: 0
Duplicate item-store-date keys: 0
